# W6D1 — ViT: Patches, Tokens and Attention Maps — Guided

**Week 6 · Day 1 · Representation Learning** · Lab

Last Thursday you built a transformer block and asserted its shapes. This afternoon you discover
that a Vision Transformer is that same block with a different tokeniser in front of it — and you
prove the tokeniser is a `Conv2d` you have already met, not a new kind of layer.

You start on the 4×4 image from this morning, cut it into four 2×2 patches by hand, and reproduce
**`[1, 2, 0, 3]`** and the cost arithmetic — **2,517,630,976** attention scores as pixels against
**38,416** as patches, a factor of **65,536**. Then you run a real pretrained ViT, confirm its
sequence length is **197** and not 196, and pull its attention maps out onto real photographs.

The `patchify` you write in task 2.1 is imported by Wednesday's lab and Thursday's. Today is where
the week's tooling gets built.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٦ · اليوم ١ — محوّل الرؤية: رقع ورموز وخرائط انتباه

**الأسبوع السادس · اليوم الأول · تعلّم التمثيل** · معمل

بنيت الخميس الماضي كتلة محوّل وفحصت أشكالها. وتكتشف بعد ظهر اليوم أن محوّل الرؤية
(Vision Transformer) هو الكتلة نفسها بمُجزّئٍ مختلف أمامها — وتُبرهن أن هذا المُجزّئ طبقة `Conv2d`
سبق أن قابلتها، لا نوعًا جديدًا من الطبقات.

تبدأ من صورة ٤×٤ التي رأيتها هذا الصباح، فتقصّها إلى أربع رقع ٢×٢ بيدك، وتُعيد إنتاج
**`[1, 2, 0, 3]`** وحساب الكلفة — **٢٬٥١٧٬٦٣٠٬٩٧٦** درجة انتباه بالبكسلات مقابل **٣٨٬٤١٦** بالرقع،
أي بعامل **٦٥٬٥٣٦**. ثم تُشغّل محوّل رؤية مُدرَّبًا مسبقًا، وتتأكّد أن طول متتاليته **١٩٧** لا ١٩٦،
وتستخرج خرائط انتباهه على صور حقيقية.

ودالة `patchify` التي تكتبها في المهمة ٢٫١ يستوردها معمل الأربعاء ومعمل الخميس. فاليوم هو اليوم
الذي تُبنى فيه أدوات الأسبوع.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Cut an image into patches with a reshape, and say what each axis of that reshape means.
- Compute what patching buys, in attention scores, for any image size and patch size.
- Show that a ViT's patch embedding is a `Conv2d` whose stride equals its kernel size, by reading
  the module list rather than by being told.
- Run a pretrained ViT, separate the `[CLS]` output from the 196 patch outputs, and say why the
  sequence is 197 long.
- Extract attention weights, average them over heads, and overlay the `[CLS]` row on the image.
- Say what an attention map is evidence of, and what it is not evidence of.
- Predict what happens when you feed a ViT the wrong input size, and explain it through the
  positional embeddings.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقصّ صورة إلى رقع بإعادة تشكيل واحدة، وأن تقول ما معنى كل محور فيها.
- أن تحسب ما تشتريه الرقع من درجات انتباه لأي مقاس صورة وأي مقاس رقعة.
- أن تُظهر أن تضمين الرقع في محوّل الرؤية طبقة `Conv2d` خطوتها تساوي نواتها، بقراءة قائمة الوحدات
  لا بالتلقين.
- أن تُشغّل محوّل رؤية مُدرَّبًا مسبقًا، وتفصل خرج `[CLS]` عن خرج الرقع الـ١٩٦، وتقول لماذا طول
  المتتالية ١٩٧.
- أن تستخرج أوزان الانتباه، وتُوسّطها على الرؤوس، وتُسقط صفّ `[CLS]` على الصورة.
- أن تقول على أي شيء تدلّ خريطة الانتباه، وعلى أي شيء لا تدلّ.
- أن تتنبّأ بما يحدث حين تُطعم المحوّل مقاسًا خاطئًا، وأن تشرحه عبر تضمينات الموضع.

</div>

## About the data

**Two datasets, and one of them is four numbers by four numbers.**

`small_image_5class` — the 400 images of weeks 4 and 6, 224×224 RGB, five classes
(`bus, cat, dog, pizza, zebra`) cut from COCO's CC-BY pool. You met them in W4D4 and W4D5. They are
here again all week, and that is deliberate: on Thursday a linear probe on frozen features gets
compared against W4D5's fine-tune **on the identical images and the identical split**, and the
comparison is void the moment the data differs.

`sample_photos` — the scene photographs from W4D1 and W4D3. Several objects each, uncropped, which
is what an attention map needs: a picture with one centred object tells you nothing about where the
model looked.

**One known problem, carried from week 4:** twelve of the eighty `pizza` images are near-duplicates
of another pizza in the same class. A checksum finds nothing. It does not bite today — nothing here
is trained — but it is the reason Thursday reuses W4D5's duplicate-grouped split rather than making
a fresh one.

**First run downloads** `google/vit-base-patch16-224` — 346 MB of ImageNet-pretrained weights,
cached afterwards. One forward pass is about **0.4 s** on a laptop CPU, and the whole notebook is
under 90 seconds of compute; the time in this lab goes on reading, not waiting.

<div dir="rtl" align="right">

## عن البيانات

**مجموعتان، وإحداهما أربعة أعداد في أربعة.**

`small_image_5class` — صور الأسبوعين الرابع والسادس الأربعمئة، ٢٢٤×٢٢٤ بالألوان، خمس فئات
(`bus, cat, dog, pizza, zebra`) مقصوصة من مجموعة COCO المرخّصة. قابلتها في الأسبوع الرابع اليومين
الرابع والخامس. وهي معك طوال هذا الأسبوع عن قصد: ففي يوم الخميس يُقارن فحصٌ خطيّ على تمثيلات مجمّدة
بالضبط الدقيق للأسبوع الرابع **على الصور نفسها والتقسيم نفسه**، وتبطل المقارنة لحظة اختلاف البيانات.

`sample_photos` — صور المشاهد من الأسبوع الرابع اليومين الأول والثالث. في كل منها عدّة أجسام وبلا
قصّ، وهذا ما تحتاجه خريطة الانتباه: فصورة فيها جسم واحد في الوسط لا تقول لك أين نظر النموذج.

**مشكلة معروفة واحدة، موروثة من الأسبوع الرابع:** اثنتا عشرة من صور `pizza` الثمانين شبه مكرّرة من
صورة أخرى في الفئة نفسها. لا تجدها البصمة الرقمية. وهي لا تضرّ اليوم لأن لا تدريب هنا، لكنها سبب
إعادة استخدام يوم الخميس لتقسيم الأسبوع الرابع المُجمَّع بالتكرارات بدل صنع تقسيم جديد.

**التشغيل الأول ينزّل** النموذج `google/vit-base-patch16-224` — ‏٣٤٦ ميجابايت من الأوزان
المُدرَّبة على ImageNet، ثم يُخزَّن. والتمريرة الأمامية الواحدة نحو **٠٫٤ ثانية** على معالج حاسوب
محمول، وحساب الدفتر كله دون تسعين ثانية؛ فوقت هذا المعمل يذهب إلى القراءة لا إلى الانتظار.

</div>

## Setup

`transformers` joins the stack today and stays for the rest of the week. One argument in the model
load matters and is easy to miss — it is called out in task 2.4.

<div dir="rtl" align="right">

## الإعداد

تنضمّ `transformers` إلى الأدوات اليوم وتبقى بقيّة الأسبوع. ويهمّ وسيطٌ واحد عند تحميل النموذج
يسهل إغفاله — نُنبّه إليه في المهمة ٢٫٤.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("transformers", "torch", "torchvision", "matplotlib", "scikit-learn")
seed_everything(42)

import json
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

use_course_style()
np.set_printoptions(precision=4, suppress=True)
torch.set_grad_enabled(False)          # nothing is trained today; this makes that literal

SEED = 42
CHECKPOINT = "google/vit-base-patch16-224"
IMAGE_SIZE = 224
PATCH_SIZE = 16

IMAGE_ROOT = get_dataset_dir("small_image_5class") / "images"
PHOTO_ROOT = get_dataset_dir("sample_photos")
CLASS_IMAGES = sorted(IMAGE_ROOT.rglob("*.jpg"))
PHOTOS = sorted(p for p in PHOTO_ROOT.rglob("*.jpg"))

ATTN_DIR = ARTEFACT_DIR / "attn_maps"
ATTN_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(CLASS_IMAGES)} class images, {len(PHOTOS)} scene photos")
print(versions(), "| device:", device())

## Section 1 — Warm-up: the 4×4 image, by hand  (≈25 min)

Everything here works. This is this morning's image:

```
1  2 | 0  1
0  3 | 1  0
-----+-----
2  1 | 4  2
1  0 | 2  3
```

Four 2×2 patches. Patch 1 is `1, 2` on top and `0, 3` below, so read in order it is
**`[1, 2, 0, 3]`** — the slide's vector. The reshape that produces all four is the only piece of
index gymnastics in the lab, and the cell below spells out what each axis is before it collapses
them.

Then the projection: the 4×3 matrix from the slide turns patch 1 into **`[0.5, 3.5, 3.5]`**. That
matrix multiply is the whole of "patch embedding".

Change one number in `IMAGE` and re-run. Watch exactly one patch vector move — and note that
shifting the image by a single column moves all four, which is slide 55's point and the reason
positional embeddings are learned per position rather than per patch.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: صورة الـ٤×٤ باليد (نحو ٢٥ دقيقة)

كل ما هنا يعمل. وهذه صورة هذا الصباح:

```
1  2 | 0  1
0  3 | 1  0
-----+-----
2  1 | 4  2
1  0 | 2  3
```

أربع رقع ٢×٢. والرقعة الأولى `1, 2` أعلى و`0, 3` أسفل، فتُقرأ بالترتيب **`[1, 2, 0, 3]`** وهو متّجه
الشريحة. وإعادة التشكيل التي تُنتج الأربع كلها هي التمرين الوحيد على الفهارس في المعمل، والخلية
أدناه تُسمّي كل محور قبل أن تطويه.

ثم الإسقاط: المصفوفة ٤×٣ التي على الشريحة تحوّل الرقعة الأولى إلى **`[0.5, 3.5, 3.5]`**. وهذا
الضرب المصفوفي هو كل ما يعنيه «تضمين الرقع».

غيّر عددًا واحدًا في `IMAGE` وأعد التشغيل. راقب متّجه رقعة واحد يتحرّك — ولاحظ أن إزاحة الصورة عمودًا
واحدًا تُحرّك الأربعة جميعًا، وهو مقصد الشريحة ٥٥ وسبب تعلّم تضمينات الموضع لكل موضع لا لكل محتوى.

</div>

In [ ]:
# This morning's image, and this morning's projection matrix.
IMAGE = np.array([[1, 2, 0, 1],
                  [0, 3, 1, 0],
                  [2, 1, 4, 2],
                  [1, 0, 2, 3]], dtype=np.float64)

W = np.array([[0.5, 0.0, 0.5],       # 4 numbers in, 3 out — the slide's patch embedding
              [0.0, 1.0, 0.0],
              [1.0, 0.0, -0.5],
              [0.0, 0.5, 1.0]])

# The reshape, one axis at a time:
#   (4, 4)          the image
#   (2, 2, 2, 2)    patch-row, row-within-patch, patch-column, column-within-patch
#   swapaxes(1, 2)  patch-row, patch-column, row-within-patch, column-within-patch
#   (4, 4)          one row per patch, each patch flattened in reading order
grid = IMAGE.reshape(2, 2, 2, 2).swapaxes(1, 2)
TOY_PATCHES = grid.reshape(4, 4)

for i, patch in enumerate(TOY_PATCHES, start=1):
    print(f"patch {i}: {patch.astype(int).tolist()}")

TOY_TOKENS = TOY_PATCHES @ W
print(f"\npatch 1 embedded: {TOY_TOKENS[0].round(2)}   (the slide says [0.5 3.5 3.5])")
print(f"all four tokens:\n{TOY_TOKENS.round(2)}")

In [ ]:
# What the patching bought, in attention scores. One head, one layer, both cases.
TOY_PIXEL_SCORES = (4 * 4) ** 2
TOY_PATCH_SCORES = 4 ** 2

REAL_PIXELS = IMAGE_SIZE * IMAGE_SIZE
REAL_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2
REAL_PIXEL_SCORES = REAL_PIXELS ** 2
REAL_PATCH_SCORES = REAL_PATCHES ** 2
RATIO = REAL_PIXEL_SCORES // REAL_PATCH_SCORES

print(f"toy  — as 16 pixels: {TOY_PIXEL_SCORES:,} scores | as 4 patches: {TOY_PATCH_SCORES:,}")
print(f"real — as {REAL_PIXELS:,} pixels: {REAL_PIXEL_SCORES:,} scores")
print(f"real — as {REAL_PATCHES} patches:  {REAL_PATCH_SCORES:,} scores")
print(f"\nratio: {REAL_PIXEL_SCORES:,} / {REAL_PATCH_SCORES:,} = {RATIO:,}")
print(f"exact: {REAL_PIXEL_SCORES % REAL_PATCH_SCORES == 0}  ({RATIO} = 2^{int(np.log2(RATIO))})")
print("\nSixty-five thousand times cheaper, for one head in one layer. That is why 16.")

## Section 2 — Core: six tasks  (≈60 min)

1. `patchify(image, patch_size)`, tested on the 4×4 and then on a real 224×224 image.
2. Load a pretrained ViT and find the patch embedding in its module list.
3. One forward pass: the `[CLS]` output, the 196 patch outputs, and why 197.
4. Attention maps on three scene photographs.
5. Four heads, four different maps, one caution.
6. Feed it 160×160 and record what happens.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. `patchify(image, patch_size)`، مفحوصة على الـ٤×٤ ثم على صورة ٢٢٤×٢٢٤ حقيقية.
٢. حمّل محوّل رؤية مُدرَّبًا مسبقًا وجِد تضمين الرقع في قائمة وحداته.
٣. تمريرة أمامية واحدة: خرج `[CLS]`، وخرج الرقع الـ١٩٦، ولماذا ١٩٧.
٤. خرائط انتباه على ثلاث صور مشاهد.
٥. أربعة رؤوس، وأربع خرائط مختلفة، وتحذير واحد.
٦. أطعمه ١٦٠×١٦٠ وسجّل ما يحدث.

</div>

### Task 2.1 — `patchify`, and the same answer from `nn.Unfold`

Write the general function. Given an image as `(H, W, C)` and a patch size, return
`(n_patches, patch_dim)` where `patch_dim = patch_size × patch_size × C`, with the patches in
reading order — left to right, then top to bottom.

Three checks, in increasing severity:

1. On the 4×4 toy image (as `(4, 4, 1)`) it must reproduce the warm-up's four vectors **exactly**.
2. On a real 224×224 RGB image it must give `(196, 768)`. That 768 is `16 × 16 × 3` and it is a
   coincidence that ViT-Base's hidden width is also 768 — the two numbers are unrelated and it is
   worth saying out loud, because a great many people conflate them.
3. It must agree with `torch.nn.Unfold`, which is the library function that does this. You wrote
   yours to know what it does; you check against theirs to know that you were right.

<div dir="rtl" align="right">

### المهمة ٢٫١ — `patchify`، والجواب نفسه من `nn.Unfold`

اكتب الدالة العامة. بمعطى صورة بالشكل `(H, W, C)` ومقاس رقعة، أعِد `(n_patches, patch_dim)` حيث
`patch_dim = patch_size × patch_size × C`، والرقع بترتيب القراءة — يمينًا ثم أسفل.

ثلاثة فحوص متصاعدة الشدّة:

١. على صورة الـ٤×٤ (بالشكل `(4, 4, 1)`) يجب أن تُعيد متّجهات الإحماء الأربعة **تمامًا**.
٢. على صورة ٢٢٤×٢٢٤ ملوّنة حقيقية يجب أن تُعطي `(196, 768)`. والـ٧٦٨ هنا هي `16 × 16 × 3`، وكونها
   أيضًا عرض ViT-Base مصادفة: العددان لا علاقة لأحدهما بالآخر، ويستحقّ هذا أن يُقال صراحةً لأن
   كثيرين يخلطون بينهما.
٣. يجب أن تتّفق مع `torch.nn.Unfold`، وهي دالة المكتبة التي تفعل ذلك. كتبتَ دالتك لتعرف ما تفعله،
   وتفحصها بدالتهم لتعرف أنك أصبت.

</div>

In [ ]:
def patchify(image, patch_size):
    """Cut an (H, W, C) image into non-overlapping patches in reading order.

    Returns (n_patches, patch_size * patch_size * C). Used by W6D3 and W6D4.
    """
    # TODO: Reshape into the five-axis grid, move the two patch-index axes to the front, and flatten. Raise a readable error when the size is not divisible by the patch size.
    # مهمة: أعِد التشكيل إلى الشبكة ذات المحاور الخمسة، وقدّم محورَي فهرس الرقعة، ثم اطوِ. وارفع خطأً مفهومًا حين لا يقبل المقاس القسمة على مقاس الرقعة.


# TODO: Check patchify three ways: against the warm-up's toy patches, on a real 224x224 image for a (196, 768) shape, and against torch.nn.Unfold on the same real image.
# مهمة: افحص `patchify` بثلاث طرق: مقابل رقع الإحماء، وعلى صورة ٢٢٤×٢٢٤ حقيقية بحثًا عن الشكل `(196, 768)`، ومقابل `torch.nn.Unfold` على الصورة نفسها.

### Task 2.2 — load the model, and find the patch embedding

Load `google/vit-base-patch16-224` and look at what the first layer actually is.

The claim from this morning was that a patch embedding *is* a strided convolution — not "is like",
not "can be implemented as". Check it: print the module, and print its kernel size and stride. If
kernel size equals stride, the convolution's windows do not overlap, and a non-overlapping window
of 16×16 that produces one vector per window is exactly the reshape-then-project you wrote in the
warm-up.

Print the parameter count too. Most of a ViT is not the tokeniser.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — حمّل النموذج، وجِد تضمين الرقع

حمّل `google/vit-base-patch16-224` وانظر ما هي أول طبقة فيه فعلًا.

كان ادّعاء هذا الصباح أن تضمين الرقع **هو** التفاف بخطوة — لا «يشبه» ولا «يمكن تنفيذه بـ». تحقّق:
اطبع الوحدة، واطبع مقاس نواتها وخطوتها. فإن ساوت الخطوةُ النواةَ لم تتداخل نوافذ الالتفاف، ونافذة
غير متداخلة ١٦×١٦ تُنتج متّجهًا واحدًا لكل نافذة هي بالضبط إعادة التشكيل ثم الإسقاط التي كتبتها في
الإحماء.

واطبع عدد المعاملات أيضًا. فمعظم محوّل الرؤية ليس المُجزّئ.

</div>

In [ ]:
from transformers import AutoImageProcessor, ViTModel

# TODO: Load the processor and the model, put the model in eval mode, then print the patch embedding module and whether its kernel size equals its stride.
# مهمة: حمّل المعالج والنموذج، وضع النموذج في وضع التقييم، ثم اطبع وحدة تضمين الرقع وهل تساوي نواتها خطوتها.

### Task 2.3 — one forward pass, and the 197th token

Run one image through and look at the output shape. It is `(1, 197, 768)`.

**196 of those are patches. The 197th is `[CLS]`** — a learned vector prepended to the sequence,
belonging to no patch, whose job is to be a place for the model to accumulate a summary of the
whole image. It is the same `[CLS]` you met in W5D5 with BERT, doing the same job, and when anyone
says "the ViT's image embedding" they mean row 0 of this tensor.

Separate the two. Confirm the shapes. Then answer, in the markdown cell below, why the sequence is
197 and not 196 — in your own words, before you read the answer that follows it.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — تمريرة أمامية واحدة، والرمز السابع والتسعون بعد المئة

مرّر صورة واحدة وانظر شكل الخرج. إنه `(1, 197, 768)`.

**١٩٦ منها رقع. والسابع والتسعون بعد المئة هو `[CLS]`** — متّجه مُتعلَّم يُوضع في صدر المتتالية، لا
ينتمي إلى رقعة، ووظيفته أن يكون مكانًا يجمع فيه النموذج خلاصة الصورة كلها. وهو `[CLS]` نفسه الذي
قابلته في الأسبوع الخامس اليوم الخامس مع BERT يؤدّي الوظيفة نفسها، وحين يقول أحدهم «تمثيل الصورة في
محوّل الرؤية» فهو يعني الصف صفر من هذا الموتّر.

افصل الاثنين. وتأكّد من الأشكال. ثم أجب في خلية Markdown أدناه: لماذا طول المتتالية ١٩٧ لا ١٩٦ —
بكلماتك، قبل أن تقرأ الجواب الذي يليها.

</div>

In [ ]:
DEMO_IMAGE = Image.open(CLASS_IMAGES[0]).convert("RGB")

# TODO: Run one image through the model, split [CLS] from the patch tokens, and print the sequence length together with both shapes.
# مهمة: مرّر صورة واحدة في النموذج، وافصل `[CLS]` عن رموز الرقع، واطبع طول المتتالية مع الشكلين.

**Why 197 and not 196.** The 196 patch tokens each describe one 16×16 square. None of them
describes the image. You could average them — people do, and it works — but averaging is a fixed
rule chosen by you, while `[CLS]` is a slot the model was *trained* to fill: it attends to whatever
patches matter and the classification head reads only that row. So the sequence carries one token
per patch plus one token for the picture, and 196 + 1 is 197.

Two consequences you will use this week: the attention maps in task 2.4 are the `[CLS]` row of the
attention matrix, which is literally "what the summary token looked at"; and Thursday's linear
probe is fitted on `[CLS]` alone — 768 numbers per image, not 196 × 768.

<div dir="rtl" align="right">

**لماذا ١٩٧ لا ١٩٦.** كل رمز من رموز الرقع الـ١٩٦ يصف مربّعًا ١٦×١٦ واحدًا. ولا واحد منها يصف
الصورة. يمكنك أن تُوسّطها — وهذا يُفعل ويعمل — لكن التوسيط قاعدة ثابتة اخترتَها أنت، بينما `[CLS]`
خانة **دُرِّب** النموذج على ملئها: تنتبه إلى ما يهمّ من الرقع، ولا يقرأ رأس التصنيف غيرها. فالمتتالية
تحمل رمزًا لكل رقعة ورمزًا للصورة، و‏١٩٦ + ١ = ١٩٧.

ولهذا نتيجتان تستعملهما هذا الأسبوع: خرائط الانتباه في المهمة ٢٫٤ هي صفّ `[CLS]` من مصفوفة الانتباه،
أي حرفيًا «إلى أين نظر رمز الخلاصة»؛ والفحص الخطيّ يوم الخميس يُلائَم على `[CLS]` وحده — ‏٧٦٨ عددًا
لكل صورة لا ‏١٩٦ × ٧٦٨.

</div>

### Task 2.4 — attention maps on real photographs

Now pull the attention weights out and look at them.

The model returns them only if you ask twice: `output_attentions=True` on the call, **and**
`attn_implementation="eager"` when you load the model. The fast fused attention kernels never
materialise the weight matrix — there is nothing to hand back — so on a default load the field
comes back `None` and it looks like a bug in your code. It is the single most common way this
task goes wrong.

The pipeline, for the last block:

1. Take the attention tensor: `(batch, heads, 197, 197)`.
2. Average over the 12 heads.
3. Take row 0 — the `[CLS]` row — and drop its first entry, which is `[CLS]` attending to itself.
4. That leaves 196 numbers. Reshape to 14×14, upsample to 224×224, and overlay.

Do it for three `sample_photos`, save each overlay to `attn_maps/`, and look at where the bright
regions land.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — خرائط الانتباه على صور حقيقية

استخرج الآن أوزان الانتباه وانظر إليها.

لا يعيدها النموذج إلا إن طلبتَها مرّتين: `output_attentions=True` عند النداء، **و**
`attn_implementation="eager"` عند التحميل. فنوى الانتباه المدمجة السريعة لا تُنشئ مصفوفة الأوزان
أصلًا — فليس ثمّة ما يُعاد — وعلى التحميل الافتراضي يعود الحقل `None` فيبدو كأنه خلل في كودك. وهذه
أكثر طريقة تفشل بها هذه المهمة.

والمسار، للكتلة الأخيرة:

١. خذ موتّر الانتباه: `(batch, heads, 197, 197)`.
٢. وسّط على الرؤوس الاثني عشر.
٣. خذ الصف صفر — صفّ `[CLS]` — واحذف مدخلته الأولى، وهي انتباه `[CLS]` إلى نفسه.
٤. يبقى ١٩٦ عددًا. أعِد تشكيلها ١٤×١٤، وكبّرها إلى ٢٢٤×٢٢٤، وأسقطها على الصورة.

افعل ذلك لثلاث صور من `sample_photos`، واحفظ كل إسقاط في `attn_maps/`، وانظر أين تقع المناطق
المضيئة.

</div>

In [ ]:
# TODO: Write cls_attention(image) returning the 14x14 map and the full [CLS] row, then overlay it on three sample_photos and save each figure into attn_maps/.
# مهمة: اكتب `cls_attention(image)` تُعيد خريطة ١٤×١٤ وصفّ `[CLS]` كاملًا، ثم أسقطها على ثلاث صور من `sample_photos` واحفظ كل شكل في `attn_maps/`.

### Task 2.5 — four heads, four maps

The average you just took hid something. Display four individual heads from the same block on the
same image and look at how different they are: the flattest spreads its weight almost evenly over
the 196 patches, while the peakiest puts roughly twice as much on its single best patch. Print the
share each head gives back to `[CLS]` itself too — in this checkpoint's last block it is
essentially nothing, which is worth knowing before you assume the map is missing weight.

Then the caution, and it is the same one from W5D3, restated because a picture makes it much easier
to forget: **an attention weight is a routing coefficient, not an explanation.** It tells you which
tokens were mixed into which. It does not tell you the model used that information, or why it
predicted what it predicted. A map that lands on the dog is consistent with the model recognising
the dog and equally consistent with it recognising the grass and routing through the dog's tokens
to get there. Attention maps are a debugging aid and a presentation aid; they are not evidence.

Write your one sentence in the markdown cell after the code.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — أربعة رؤوس، أربع خرائط

أخفى التوسيط الذي فعلتَه شيئًا. اعرض أربعة رؤوس منفردة من الكتلة نفسها على الصورة نفسها وانظر كم
تختلف: أقلّها تركيزًا يوزّع وزنه شبه المتساوي على الرقع الـ١٩٦، وأشدّها تركيزًا يضع على أفضل رقعة
عنده نحو ضعف ذلك. واطبع أيضًا نصيب `[CLS]` نفسه من كل رأس — وهو في الكتلة الأخيرة من هذا النموذج
لا يكاد يُذكر، ويحسن أن تعرف ذلك قبل أن تظنّ أن الخريطة فقدت وزنًا.

ثم التحذير، وهو نفسه تحذير الأسبوع الخامس اليوم الثالث، نُعيده لأن الصورة تجعل نسيانه أسهل بكثير:
**وزن الانتباه معامل توجيه لا تفسير.** يخبرك أي الرموز مُزجت في أيّها. ولا يخبرك أن النموذج استعمل
تلك المعلومة، ولا لماذا تنبّأ بما تنبّأ. فخريطة تقع على الكلب تتّسق مع تعرّف النموذج على الكلب،
وتتّسق بالقدر نفسه مع تعرّفه على العشب ومروره برموز الكلب في الطريق. خرائط الانتباه عون على التنقيح
وعون على العرض؛ وليست دليلًا.

اكتب جملتك في خلية Markdown بعد الكود.

</div>

In [ ]:
# TODO: Display heads 0, 3, 6 and 9 for one photo side by side, and print each head's maximum patch weight and how much weight it put on [CLS] itself.
# مهمة: اعرض الرؤوس ٠ و٣ و٦ و٩ لصورة واحدة جنبًا إلى جنب، واطبع لكل رأس أكبر وزن رقعة ومقدار ما وضعه على `[CLS]` نفسه.

**Your sentence.** Replace this line with one sentence naming a difference you can see between two
of the four heads, and one thing the maps do **not** prove.

*(Reference answer, for after you have written yours: head 3 puts about twice the weight of head 6
on its single strongest patch, so one is reading a region and the other is reading the whole frame
— and neither map proves the model used those patches to decide, only that information was routed
through them.)*

<div dir="rtl" align="right">

**جملتك.** استبدل هذا السطر بجملة واحدة تسمّي فرقًا تراه بين اثنين من الرؤوس الأربعة، وشيئًا واحدًا
**لا** تُبرهنه الخرائط.

*(جواب مرجعي بعد أن تكتب جوابك: يضع الرأس ٣ على أقوى رقعة عنده نحو ضعف ما يضعه الرأس ٦، فأحدهما
يقرأ منطقة والآخر يقرأ الإطار كلّه — ولا تُبرهن أيّ خريطة أن النموذج استعمل تلك الرقع في قراره،
إنما تُظهر أن المعلومة مرّت بها.)*

</div>

### Task 2.6 — the resolution trap

Feed the model a 160×160 image instead of 224×224 and record what happens.

There are two possible outcomes and which one you get depends on the checkpoint. Either it raises,
or it silently produces an output that is worse than it should be. Find out which, capture the
evidence, and then explain it in terms of **positional embeddings**: this checkpoint learned
`196 + 1` position vectors, one per grid cell of a 14×14 grid. A 160×160 image at patch 16 is a
10×10 grid — 100 patches. There is no position vector 101 through 196 to leave out, and no vector
for "cell (3, 7) of a 10×10 grid" either, because the ones it has mean "cell (3, 7) of a 14×14
grid" and those are different places on the picture.

Interpolating the position embeddings is the standard fix and it works; `interpolate_pos_encoding`
is the flag. Try it, and note that the model runs but the sequence length changes.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — فخّ المقاس

أطعِم النموذج صورة ١٦٠×١٦٠ بدل ٢٢٤×٢٢٤ وسجّل ما يحدث.

هناك احتمالان، وأيّهما تحصل عليه يعتمد على النموذج المحفوظ: إما أن يرفع استثناءً، وإما أن يُخرج
صامتًا نتيجةً أسوأ ممّا ينبغي. اعرف أيّهما، والتقط الدليل، ثم اشرحه عبر **تضمينات الموضع**: تعلّم
هذا النموذج `196 + 1` متّجه موضع، واحدًا لكل خلية في شبكة ١٤×١٤. وصورة ١٦٠×١٦٠ برقعة ١٦ شبكتها
١٠×١٠ أي مئة رقعة. فلا يوجد متّجه موضع من ١٠١ إلى ١٩٦ لتحذفه، ولا متّجه لـ«الخلية (٣، ٧) من شبكة
١٠×١٠»، لأن ما عنده يعني «الخلية (٣، ٧) من شبكة ١٤×١٤» وهما موضعان مختلفان على الصورة.

واستيفاء تضمينات الموضع هو الإصلاح المعتاد وهو يعمل؛ والوسيط اسمه `interpolate_pos_encoding`.
جرّبه، ولاحظ أن النموذج يعمل لكن طول المتتالية يتغيّر.

</div>

In [ ]:
SMALL_SIZE = 160

# TODO: Feed the model a SMALL_SIZE image without the processor's resize, record whether it raised or ran, then repeat with interpolate_pos_encoding=True.
# مهمة: أطعِم النموذج صورة بمقاس `SMALL_SIZE` دون إعادة تحجيم المعالج، وسجّل هل رفع استثناءً أم عمل، ثم أعِد الكرّة بـ`interpolate_pos_encoding=True`.

## Section 3 — Stretch: does the ViT agree with the CNN?  (≈30 min)

Two models, twenty images, one question.

Take twenty images from `small_image_5class`. Get the ViT's `[CLS]` vector for each — 768 numbers.
Get ResNet-18's penultimate-layer vector for each — 512 numbers, the same features W4D5 froze and
put a linear head on. The two vectors are not comparable to each other; **the similarity matrices
are.**

Build both 20×20 cosine matrices, display them side by side, and correlate their off-diagonal
entries. If the two models organise images the same way, the correlation is high and the choice
between them is about cost. If it is low, they are noticing different things, and Thursday's
comparison table is measuring something real rather than two paths to the same answer.

Then answer, in writing: **on 400 images, would you pick the ViT or the CNN, and what evidence do
you have from today?** There is a defensible answer in both directions and the evidence matters
more than the choice. Note what you do *not* yet know — accuracy, for one, which is Thursday.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: هل يتّفق محوّل الرؤية مع الشبكة الالتفافية؟ (نحو ٣٠ دقيقة)

نموذجان، وعشرون صورة، وسؤال واحد.

خذ عشرين صورة من `small_image_5class`. واحصل على متّجه `[CLS]` من محوّل الرؤية لكل منها — ٧٦٨ عددًا.
واحصل على متّجه الطبقة قبل الأخيرة من ResNet-18 لكل منها — ٥١٢ عددًا، وهي التمثيلات نفسها التي
جمّدها الأسبوع الرابع اليوم الخامس ووضع فوقها رأسًا خطيًا. والمتّجهان غير قابلين للمقارنة أحدهما
بالآخر؛ **لكن مصفوفتَي التشابه قابلتان.**

ابنِ مصفوفتَي جيب تمام ٢٠×٢٠، واعرضهما جنبًا إلى جنب، وارتبط بين مدخلاتهما خارج القطر. فإن نظّم
النموذجان الصور التنظيم نفسه كان الارتباط عاليًا وصار الاختيار بينهما مسألة كلفة. وإن كان منخفضًا
فهما يلحظان أشياء مختلفة، وجدول الخميس يقيس شيئًا حقيقيًا لا طريقين إلى الجواب نفسه.

ثم أجب كتابةً: **على أربعمئة صورة، أتختار محوّل الرؤية أم الشبكة الالتفافية، وما دليلك من اليوم؟**
ثمّة جواب يُدافَع عنه في الاتجاهين، والدليل أهمّ من الاختيار. وسمِّ ما **لا** تعرفه بعد — الدقّة
مثلًا، وهي يوم الخميس.

</div>

In [ ]:
from torchvision import models

# TODO: Extract ViT [CLS] and ResNet-18 penultimate features for the same 20 images, build both cosine matrices, plot them side by side, and correlate their off-diagonal entries.
# مهمة: استخرج `[CLS]` من المحوّل وتمثيلات ما قبل الأخيرة من ResNet-18 للعشرين صورة نفسها، وابنِ مصفوفتَي جيب التمام، وارسمهما جنبًا إلى جنب، وارتبط بين مدخلاتهما خارج القطر.

**Your answer.** Two or three sentences: ViT or CNN on 400 images, the evidence from today, and
one thing you would need to measure before committing.

*(A note on the ResNet numbers you just saw: its cosine similarities sit much higher and in a much
narrower band than the ViT's. That is a property of ReLU features — they are non-negative, so every
pair of vectors already points into the same orthant — not a sign that the ResNet finds everything
similar. Comparing the two mean similarities directly is therefore meaningless; comparing the
*structure*, which is what the correlation does, is not.)*

<div dir="rtl" align="right">

**جوابك.** جملتان أو ثلاث: المحوّل أم الشبكة الالتفافية على أربعمئة صورة، والدليل من اليوم، وشيء
واحد تحتاج إلى قياسه قبل الالتزام.

*(ملاحظة على أرقام ResNet التي رأيتها: جيوب تمامها أعلى بكثير وفي نطاق أضيق بكثير من نظيرتها في
المحوّل. وهذه خاصّية تمثيلات ReLU — فهي غير سالبة، فكل زوج متّجهات يشير أصلًا إلى الثُّمن نفسه — لا
دليلٌ على أن الشبكة ترى كل شيء متشابهًا. فمقارنة المتوسّطين مباشرةً بلا معنى، أما مقارنة **البنية**
وهي ما يفعله الارتباط فليست كذلك.)*

</div>

## Save your artefact

Two things go to disk, and both are loaded later in the week.

`patch_check.json` — the four hand-verified patch vectors, the cost arithmetic, the sequence
length, and what the 160×160 image did. **Wednesday's warm-up reloads this file**, masks three of
your four toy patches, and asks you to guess the missing values before an MAE does it for you.

`w6_patches.py` — your `patchify`, written out as an importable module. Wednesday and Thursday
import it rather than redefining it. If you skip today, `load_artefact` falls back to the reference
copy in `solutions_cache` and you are not blocked — but the version on disk should be yours.

<div dir="rtl" align="right">

## احفظ أثرك

شيئان يذهبان إلى القرص، ويُحمَّل كلاهما لاحقًا هذا الأسبوع.

`patch_check.json` — متّجهات الرقع الأربعة المُتحقَّق منها يدويًا، وحساب الكلفة، وطول المتتالية، وما
فعلته صورة الـ١٦٠×١٦٠. **وإحماء الأربعاء يُعيد تحميل هذا الملف**، فيُخفي ثلاثًا من رقعك الأربع
ويطلب منك تخمين القيم الغائبة قبل أن يفعلها المُرمِّز المُقنَّع نيابةً عنك.

`w6_patches.py` — دالة `patchify` عندك مكتوبةً وحدةً قابلة للاستيراد. ويستوردها الأربعاء والخميس بدل
إعادة تعريفها. وإن تغيّبت اليوم رجع `load_artefact` إلى النسخة المرجعية في `solutions_cache` فلا
تتعطّل — لكن الأولى أن تكون النسخة التي على القرص نسختك.

</div>

In [ ]:
import inspect

patch_check = {
    "toy_image": IMAGE.astype(int).tolist(),
    "toy_patches": TOY_PATCHES.astype(int).tolist(),
    "toy_tokens": TOY_TOKENS.round(4).tolist(),
    "toy_pixel_scores": TOY_PIXEL_SCORES,
    "toy_patch_scores": TOY_PATCH_SCORES,
    "real_pixel_scores": REAL_PIXEL_SCORES,
    "real_patch_scores": REAL_PATCH_SCORES,
    "ratio": int(RATIO),
    "checkpoint": CHECKPOINT,
    "sequence_length": int(SEQUENCE_LENGTH),
    "n_patches": int(REAL_PATCHES),
    "hidden_size": int(CLS_TOKEN.shape[0]),
    "patch_dim": int(REAL_PATCH_ARRAY.shape[1]),
    "forward_seconds": round(float(FORWARD_SECONDS), 3),
    "resolution_trap": {"size": SMALL_SIZE, "outcome": RESOLUTION_OUTCOME,
                        "evidence": RESOLUTION_EVIDENCE,
                        "interpolated_length": int(INTERPOLATED_LENGTH)},
    "vit_resnet_agreement": round(AGREEMENT, 4),
}

CHECK_PATH = ARTEFACT_DIR / "patch_check.json"
CHECK_PATH.write_text(json.dumps(patch_check, indent=2), encoding="utf-8")

MODULE_PATH = ARTEFACT_DIR / "w6_patches.py"
MODULE_PATH.write_text(
    '"""patchify, as written in W6D1. Imported by W6D3 and W6D4."""\n\n'
    "import numpy as np\n\n\n" + inspect.getsource(patchify),
    encoding="utf-8")

print(json.dumps({k: v for k, v in patch_check.items() if k != "toy_tokens"}, indent=2))
print(f"\nwrote {CHECK_PATH.name} and {MODULE_PATH.name} — Wednesday loads both")
print(f"and {len(list(ATTN_DIR.glob('*.png')))} overlays in {ATTN_DIR.name}/")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(TOY_MATCHES,
      f"patchify must reproduce the warm-up's four patch vectors exactly — got "
      f"{patchify(IMAGE[:, :, None], 2).astype(int).tolist()} against "
      f"{TOY_PATCHES.astype(int).tolist()}. Patch 1 has to read [1, 2, 0, 3]",
      f"يجب أن تُعيد `patchify` متّجهات الإحماء الأربعة تمامًا — والناتج "
      f"{patchify(IMAGE[:, :, None], 2).astype(int).tolist()} مقابل "
      f"{TOY_PATCHES.astype(int).tolist()}. والرقعة الأولى يجب أن تُقرأ `[1, 2, 0, 3]`")

check(REAL_PATCH_ARRAY.shape == (196, 768) and UNFOLD_AGREES,
      f"a 224x224 RGB image must give exactly (196, 768) patches and agree with nn.Unfold — got "
      f"{REAL_PATCH_ARRAY.shape}, agrees: {UNFOLD_AGREES}. A mismatch with Unfold is almost always "
      f"the channel axis in the wrong place",
      f"يجب أن تُعطي صورة ٢٢٤×٢٢٤ ملوّنة الشكل `(196, 768)` تمامًا وأن تتّفق مع `nn.Unfold` — والناتج "
      f"{REAL_PATCH_ARRAY.shape}، والاتّفاق: {UNFOLD_AGREES}. وعدم الاتّفاق سببه غالبًا محور القنوات "
      f"في غير موضعه")

check(RATIO == 65_536 and REAL_PIXEL_SCORES == 2_517_630_976,
      f"the cost arithmetic must reproduce the lecture: {REAL_PIXEL_SCORES:,} pixel scores and a "
      f"ratio of exactly 65,536 — got {REAL_PIXEL_SCORES:,} and {RATIO:,}",
      f"يجب أن يُعيد حساب الكلفة أرقام المحاضرة: {REAL_PIXEL_SCORES:,} درجة بالبكسلات ونسبة "
      f"‏٦٥٬٥٣٦ تمامًا — والناتج {REAL_PIXEL_SCORES:,} و{RATIO:,}")

check(SEQUENCE_LENGTH == 197 and STRIDE_EQUALS_KERNEL,
      f"the model's sequence length must be 197 (196 patches + [CLS]) and its patch embedding must "
      f"be a Conv2d with stride == kernel — got length {SEQUENCE_LENGTH}, stride==kernel "
      f"{STRIDE_EQUALS_KERNEL}",
      f"يجب أن يكون طول متتالية النموذج ١٩٧ (١٩٦ رقعة + `[CLS]`) وأن يكون تضمين الرقع `Conv2d` "
      f"خطوته تساوي نواته — والطول {SEQUENCE_LENGTH}، والخطوة تساوي النواة {STRIDE_EQUALS_KERNEL}")

check(all(abs(row.sum() - 1.0) < 1e-4 for row in ATTENTION_ROWS.values()),
      f"every [CLS] attention row must sum to 1 across the 197 tokens — got "
      f"{[round(float(r.sum()), 6) for r in ATTENTION_ROWS.values()]}. If a row sums to 12 you "
      f"summed the heads instead of averaging them",
      f"يجب أن يجمع كل صفّ انتباه لـ`[CLS]` إلى واحد على الرموز الـ١٩٧ — والناتج "
      f"{[round(float(r.sum()), 6) for r in ATTENTION_ROWS.values()]}. وإن جمع صفٌّ إلى ١٢ فقد "
      f"جمعتَ الرؤوس بدل توسيطها")

check(len(list(ATTN_DIR.glob("*.png"))) >= 3
      and all(Image.open(p).size >= (224, 224) for p in ATTN_DIR.glob("*.png")),
      f"attn_maps/ must hold at least three overlays, each at least 224x224 — found "
      f"{[(p.name, Image.open(p).size) for p in sorted(ATTN_DIR.glob('*.png'))]}",
      f"يجب أن يحوي `attn_maps/` ثلاثة إسقاطات على الأقل، كلٌّ منها ٢٢٤×٢٢٤ فأكثر — والموجود "
      f"{[(p.name, Image.open(p).size) for p in sorted(ATTN_DIR.glob('*.png'))]}")

check(RESOLUTION_OUTCOME in {"ran", "raised"} and len(RESOLUTION_EVIDENCE) > 10,
      f"the 160x160 case must be recorded either way — outcome {RESOLUTION_OUTCOME!r}, evidence "
      f"{RESOLUTION_EVIDENCE!r}. The check is that you wrote down what happened, not which way it "
      f"went",
      f"يجب تسجيل حالة ١٦٠×١٦٠ في الحالتين — النتيجة {RESOLUTION_OUTCOME!r}، والدليل "
      f"{RESOLUTION_EVIDENCE!r}. والمفحوص أنك دوّنت ما حدث لا أيّهما حدث")

report()

## What's next

**W6D2 — an autoencoder, and no labels at all.** Tomorrow is the first model in the bootcamp
trained without a single label: a conv encoder, a bottleneck, a decoder, and mean squared error
against the input itself.

The finding is a divergence. You train at four bottleneck widths, watch the reconstruction error
fall monotonically as the bottleneck widens — and then fit a classifier on the codes and watch
accuracy **collapse** at the widest one, exactly where reconstruction looks best. Reconstruction
error is not a representation score, and tomorrow proves it with two lines on one axis.

Today's `patch_check.json` is not needed until Wednesday. Leave it where it is.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٦ اليوم ٢ — مُرمِّز ذاتي، وبلا تسميات إطلاقًا.** الغد أول نموذج في المعسكر يُدرَّب دون
تسمية واحدة: مُرمِّز التفافي، وعنق زجاجة، ومُفكِّك ترميز، وخطأ تربيعي متوسّط مقابل الدخل نفسه.

والنتيجة تباعد. تُدرّب عند أربعة عروض لعنق الزجاجة، وتراقب خطأ إعادة البناء يهبط باطّراد كلّما اتّسع
العنق — ثم تُلائم مصنّفًا على الشيفرات وتراقب الدقّة **تنهار** عند أوسعها، أي حيث تبدو إعادة البناء
في أفضل حال. فخطأ إعادة البناء ليس درجةً للتمثيل، والغد يُبرهن ذلك بخطّين على محور واحد.

ولا يُحتاج `patch_check.json` قبل الأربعاء. اتركه مكانه.

</div>